In [1]:
import pandas as pd

## Ingestion

In [2]:
df = pd.read_csv('../data/feature_store_data.csv')
df = df.drop_duplicates(subset=['feature_name'])

In [3]:
df.columns

Index(['id', 'feature_name', 'feature_group', 'computation_logic',
       'data_source', 'update_frequency', 'serving_store',
       'models_using_feature', 'feature_description'],
      dtype='str')

In [4]:
import minsearch
from minsearch import Index

# Create the MinSearch index with your text and keyword fields
# Define which fields should be searchable
index = minsearch.Index(text_fields = [
    "feature_name",
    "feature_group", 
    "computation_logic",
    "data_source",
    "update_frequency",
    "serving_store",
    "models_using_feature",
    "feature_description"
], keyword_fields = ["id"])

# Convert DataFrame to list of dictionaries
documents = df.to_dict(orient="records")

# Create and fit the index
index.fit(documents)

In [5]:
query = 'what features are used for personalized promotion campaigns?'
index.search(query, num_results = 7)

[{'id': 2,
  'feature_name': 'purchase_count_30d_web',
  'feature_group': 'customer_behavior',
  'computation_logic': "COUNT(order_id) WHERE order_status='DELIVERED' OVER last 30 days BY customer_id",
  'data_source': 'fct_orders (Silver)',
  'update_frequency': 'Hourly',
  'serving_store': 'DynamoDB, S3',
  'models_using_feature': 'recommendation_model, churn_model',
  'feature_description': 'Monthly purchase count indicator. Useful for RFM segmentation, customer lifetime value modeling, and personalized promotion campaigns. Captured from the web channel, representing customer interactions on the Amazon website.'},
 {'id': 1,
  'feature_name': 'purchase_count_14d_web',
  'feature_group': 'customer_behavior',
  'computation_logic': "COUNT(order_id) WHERE order_status='DELIVERED' OVER last 14 days BY customer_id",
  'data_source': 'fct_orders (Silver)',
  'update_frequency': 'Hourly',
  'serving_store': 'DynamoDB, S3',
  'models_using_feature': 'recommendation_model, churn_model',
  'fe

## RAG flow

In [6]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load .envrc (or .env) file
load_dotenv('../.envrc')
client = OpenAI()

In [7]:
def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [8]:
documents[0]

{'id': 0,
 'feature_name': 'purchase_count_7d_web',
 'feature_group': 'customer_behavior',
 'computation_logic': "COUNT(order_id) WHERE order_status='DELIVERED' OVER last 7 days BY customer_id",
 'data_source': 'fct_orders (Silver)',
 'update_frequency': 'Hourly',
 'serving_store': 'DynamoDB, S3',
 'models_using_feature': 'recommendation_model, churn_model',
 'feature_description': 'Short-term weekly purchase count metric. Useful for RFM segmentation, customer lifetime value modeling, and personalized promotion campaigns. Captured from the web channel, representing customer interactions on the Amazon website.'}

In [9]:
prompt_template = """
You are a feature store documentation assistant for an Amazon online shopping platform.
Answer the QUESTION based on the CONTEXT from our feature store documentation.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
feature_name: {feature_name}
feature_group: {feature_group}
computation_logic: {computation_logic}
data_source: {data_source}
update_frequency: {update_frequency}
serving_store: {serving_store}
models_using_feature: {models_using_feature}
feature_description : {feature_description}
""".strip()

def build_prompt(query, search_results):
    context = ""
    
    for doc in search_results:
        context = context + entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [10]:
search_results = search(query)
prompt = build_prompt(query, search_results)

In [11]:
print(prompt)

You are a feature store documentation assistant for an Amazon online shopping platform.
Answer the QUESTION based on the CONTEXT from our feature store documentation.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: what features are used for personalized promotion campaigns?

CONTEXT:
feature_name: purchase_count_30d_web
feature_group: customer_behavior
computation_logic: COUNT(order_id) WHERE order_status='DELIVERED' OVER last 30 days BY customer_id
data_source: fct_orders (Silver)
update_frequency: Hourly
serving_store: DynamoDB, S3
models_using_feature: recommendation_model, churn_model
feature_description : Monthly purchase count indicator. Useful for RFM segmentation, customer lifetime value modeling, and personalized promotion campaigns. Captured from the web channel, representing customer interactions on the Amazon website.

feature_name: purchase_count_14d_web
feature_group: customer_behavior
computation_logic: COUNT(order_id) WHERE order_status='DEL

In [12]:
def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [13]:
def rag(query, model='gpt-4o-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    #print(prompt)
    answer = llm(prompt, model=model)
    return answer

In [14]:
answer = rag(query)
print(answer)

The features used for personalized promotion campaigns include:

1. **purchase_count_30d_web**: Monthly purchase count indicator from the web channel.
2. **purchase_count_14d_web**: Bi-weekly purchase count measurement from the web channel.
3. **purchase_count_7d_web**: Short-term weekly purchase count metric from the web channel.
4. **purchase_count_30d_mobile**: Monthly purchase count indicator from the mobile app channel.
5. **purchase_count_14d_mobile**: Bi-weekly purchase count measurement from the mobile app channel.
6. **purchase_count_7d_mobile**: Short-term weekly purchase count metric from the mobile app channel.


## Retrieval evaluation

In [15]:
# Ground truth generation for retrieval evaluation
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')

In [16]:
df_question.head()

,id,question
0,0,How can the purchase_count_7d_web feature help...
1,0,In what ways can this short-term purchase coun...
2,0,What is the frequency of updates for the purch...
3,0,Can you tell me more about the data source for...
4,0,Which models are currently utilizing the purch...


In [17]:
ground_truth = df_question.to_dict(orient='records')

In [18]:
ground_truth[10]

{'id': 2,
 'question': 'How can I leverage the purchase_count_30d_web feature to enhance my understanding of customer loyalty in our upcoming marketing campaigns?'}

In [19]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [20]:
def minsearch_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [21]:
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['id']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

In [22]:
from tqdm.auto import tqdm

In [23]:
evaluate(ground_truth, lambda q: minsearch_search(q['question']))

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.9888888888888889, 'mrr': 0.8606371252204583}

## Finding the best parameters

In [24]:
import random

In [30]:
# # Split into validation (first 150) and test (the rest)
df_val = df_question[:180]
df_test = df_question[180:]
print(f"Validation set: {len(df_val)} questions")
print(f"Test set: {len(df_test)} questions")

Validation set: 180 questions
Test set: 180 questions


In [35]:
def simple_optimize(param_ranges, objective_function, n_iterations=10):
    best_params = None
    best_score = float('-inf')  # Assuming we're minimizing. Use float('-inf') if maximizing.

    for _ in range(n_iterations):
        # Generate random parameters
        current_params = {}
        for param, (min_val, max_val) in param_ranges.items():
            if isinstance(min_val, int) and isinstance(max_val, int):
                current_params[param] = random.randint(min_val, max_val)
            else:
                current_params[param] = random.uniform(min_val, max_val)
        
        # Evaluate the objective function
        current_score = objective_function(current_params)
        
        # Update best if current is better
        if current_score > best_score:  # Change to > if maximizing
            best_score = current_score
            best_params = current_params
    
    return best_params, best_score

In [37]:
gt_val = df_val.to_dict(orient='records')

In [38]:
def minsearch_search(query, boost=None):
    if boost is None:
        boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [39]:
def objective(boost_params):
    def search_function(q):
        return minsearch_search(q['question'], boost_params)

    results = evaluate(gt_val, search_function)
    return results['mrr']

In [ ]:
param_ranges = {
    'feature_name': (0.0, 3.0),
    'feature_group': (0.0, 3.0),
    'feature_description': (0.0, 3.0),
    'computation_logic': (0.0, 2.0),
    'models_using_feature': (0.0, 2.0),
    'data_source': (0.0, 1.5),
    'serving_store': (0.0, 1.0),
    'update_frequency': (0.0, 1.0),
}

In [40]:
simple_optimize(param_ranges, objective, n_iterations=20)

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

...

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

  0%|          | 0/180 [00:00<?, ?it/s]

({'feature_name': 2.0669181510194905,
  'feature_group': 0.18270662460754072,
  'feature_description': 2.6961132544858506,
  'computation_logic': 1.9081781710655161,
  'models_using_feature': 1.1690425199776486,
  'data_source': 1.2970697169946606,
  'serving_store': 0.7658460281584135,
  'update_frequency': 0.5834003457070657},
 0.9164572310405644)

In [41]:
def minsearch_improved(query):
    boost = {
        'feature_name': 2.07,
        'feature_group': 0.18,
        'feature_description': 2.70,
        'computation_logic': 1.90,
        'models_using_feature': 1.17,
        'data_source': 1.30,
        'serving_store': 0.77,
        'update_frequency' : 0.58
    }

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

evaluate(ground_truth, lambda q: minsearch_improved(q['question']))

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.9888888888888889, 'mrr': 0.8982087742504408}

## RAG evaluation

In [56]:
prompt2_template = """
You are an expert evaluator for a feature store documentation RAG system.
Your task is to analyze the relevance of the generated answer to the given question
about features, their computation logic, data sources, and usage in machine learning models.

Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the question
and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

In [53]:
len(ground_truth)

360

In [57]:
record = ground_truth[0]
question = record['question']
answer_llm = rag(question)

In [58]:
print(answer_llm)

The purchase_count_7d_web feature helps in determining the right time to launch targeted promotions by providing a short-term weekly metric of customer purchases. By analyzing the count of delivered orders over the last 7 days for each customer, businesses can gain insights into recent shopping behavior. This information is particularly useful for RFM (Recency, Frequency, Monetary) segmentation and identifying customers who may be more receptive to personalized promotions. By timing promotions to align with increases in purchase activity, businesses can enhance the effectiveness of their marketing efforts and better engage customers.


In [59]:
prompt = prompt2_template.format(question=question, answer_llm=answer_llm)
print(prompt)

You are an expert evaluator for a feature store documentation RAG system.
Your task is to analyze the relevance of the generated answer to the given question
about features, their computation logic, data sources, and usage in machine learning models.

Based on the relevance of the generated answer, you will classify it
as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Question: How can the purchase_count_7d_web feature help in determining the right time to launch targeted promotions for our customers?
Generated Answer: The purchase_count_7d_web feature helps in determining the right time to launch targeted promotions by providing a short-term weekly metric of customer purchases. By analyzing the count of delivered orders over the last 7 days for each customer, businesses can gain insights into recent shopping behavior. This information is particularly useful for RFM (Recency, Frequency, Monetary) segmentation and identifying customers who may be mo

In [60]:
llm(prompt)

'{\n  "Relevance": "RELEVANT",\n  "Explanation": "The generated answer directly addresses how the purchase_count_7d_web feature can be utilized to time targeted promotions based on recent customer purchasing behavior. It explains the significance of the feature in conjunction with RFM segmentation, demonstrating its relevance to determining promotional timing."\n}'

In [74]:
df_sample = df_question.sample(n=200, random_state=1)

In [75]:
sample = df_sample.to_dict(orient='records')

In [76]:
evaluations = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question) 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation = llm(prompt)
    evaluation = json.loads(evaluation)

    evaluations.append((record, answer_llm, evaluation))

  0%|          | 0/200 [00:00<?, ?it/s]

In [77]:
evaluations[0]

({'id': 51,
  'question': 'How frequently is the return_rate_7d_mobile feature updated and what implications does this have for real-time decision-making in our operations?'},
 'The return_rate_7d_mobile feature is updated daily. This daily update frequency allows for relatively timely insights into product quality and customer return behaviors, enabling real-time decision-making in operations such as fraud detection, return risk analysis, and product quality monitoring. However, since it is not updated in real-time or more frequently than once per day, there may be a lag in capturing fluctuations in return rates, which could affect immediate operational adjustments.',
 {'Relevance': 'RELEVANT',
  'Explanation': 'The generated answer directly addresses the question by specifying that the return_rate_7d_mobile feature is updated daily. It also discusses the implications of this update frequency for real-time decision-making, including the benefits and potential lag in capturing immediat

In [78]:
df_eval = pd.DataFrame(evaluations, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])

df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [79]:
df_eval.relevance.value_counts(normalize=True).reindex(['RELEVANT', 'PARTLY_RELEVANT', 'NON_RELEVANT'], fill_value=0)

relevance
RELEVANT           0.99
PARTLY_RELEVANT    0.01
NON_RELEVANT       0.00
Name: proportion, dtype: float64

In [80]:
df_eval.to_csv('../data/rag-eval-gpt-4o-mini.csv', index=False)

In [81]:
df_eval[df_eval.relevance == 'PARTLY_RELEVANT']

,answer,id,question,relevance,explanation
91,We can leverage the bi-weekly purchase count t...,37,How can we leverage the bi-weekly purchase cou...,PARTLY_RELEVANT,The generated answer discusses leveraging bi-w...
109,"Monitoring the monthly click rate indicator, b...",26,In what ways can monitoring the monthly click ...,PARTLY_RELEVANT,The generated answer provides insights into ho...


In [83]:
import re

In [85]:
def parse_evaluation_response(response_text):
    """Parse LLM evaluation response with error handling."""
    try:
        # Try direct JSON parsing
        return json.loads(response_text)
    except json.JSONDecodeError:
        # Try to extract JSON from the response
        try:
            # Find JSON pattern - look for anything that starts with { and ends with }
            json_match = re.search(r'\{[^{}]*"Relevance"[^{}]*\}', response_text, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
        except:
            pass
        
        # Try to find any JSON-like structure
        try:
            start = response_text.find('{')
            end = response_text.rfind('}') + 1
            if start != -1 and end != -1:
                json_str = response_text[start:end]
                return json.loads(json_str)
        except:
            pass
        
        # Fallback: return default
        print(f"Warning: Failed to parse: {response_text[:200]}...")
        return {
            "Relevance": "UNKNOWN",
            "Explanation": f"Failed to parse evaluation. Raw response: {response_text[:100]}"
        }

In [ ]:
# Run evaluation with error handling
evaluations_gpt4o = []

for record in tqdm(sample):
    question = record['question']
    answer_llm = rag(question, model='gpt-4o') 

    prompt = prompt2_template.format(
        question=question,
        answer_llm=answer_llm
    )

    evaluation_response = llm(prompt, model='gpt-4o')
    evaluation = parse_evaluation_response(evaluation_response)
    
    evaluations_gpt4o.append({
        "id": record["id"],
        "question": question,
        "answer": answer_llm,
        "relevance": evaluation.get("Relevance", "UNKNOWN"),
        "explanation": evaluation.get("Explanation", "Failed to parse")
    })

In [ ]:
# Convert to DataFrame directly from list of dictionaries
df_eval = pd.DataFrame(evaluations_gpt4o)

In [ ]:
df_eval.relevance.value_counts(normalize=True)

In [ ]:
df_eval = pd.DataFrame(evaluations_gpt4o, columns=['record', 'answer', 'evaluation'])

df_eval['id'] = df_eval.record.apply(lambda d: d['id'])
df_eval['question'] = df_eval.record.apply(lambda d: d['question'])

df_eval['relevance'] = df_eval.evaluation.apply(lambda d: d['Relevance'])
df_eval['explanation'] = df_eval.evaluation.apply(lambda d: d['Explanation'])

del df_eval['record']
del df_eval['evaluation']

In [ ]:
df_eval.relevance.value_counts(normalize=True)

In [ ]:
df_eval.to_csv('../data/rag-eval-gpt-4o.csv', index=False)